# Explainable AI-Based Resume Scoring System

## Data Cleaning

This notebook builds a **reproducible, bias-aware cleaning pipeline** for resume text.
It standardizes text, redacts PII, filters junk, de-duplicates, extracts simple section cues, and writes auditable outputs for downstream EDA and modeling.

**Outputs**
- `data/processed/resumes_clean.csv` (+ `.parquest` if available)
- `reports/duplicates.csv`
- `reports/data_quality.json`

### Imports and Reproducibility Seeds

Load core libraries and fix randomness to make results repeatable

**Notes**
- `RANDOM_SEED` is reused later (sampling, splits, models).
- Optional deps (`spacy`, `rapidfuzz`, `unidecode`) enable extra features but the pipeline still runs without them.

**Key Params**
- `MIN_CHARS` / `MIN_TOKENS` - filters out empty/very short resumes
- Redaction toggles - `REDACT_EMAIL`, `REDACT_PHONE`, `REDACT_URL`, `REDACT_PERSON`
- `DUP_SIMILARITY` - near-duplicate similarity threshold (0-100)

In [52]:
import pandas as pd
import numpy as np
import re, json, unicodedata, hashlib
from pathlib import Path

In [53]:
try:
    from unidecode import unidecode
except Exception:
    unidecode = lambda x: x

try:
    import spacy
    try:
        _NLP = spacy.load("en_core_web_sm")
    except Exception:
        _NLP = None
except Exception:
    _NLP = None

try:
    from rapidfuzz import fuzz
    _FUZZ_OK = True
except Exception:
    _FUZZ_OK = False

In [54]:
# Config
MIN_CHARS = 100
MIN_TOKENS = 20
ENABLE_REDACTION = True
REDACT_PERSON = True
REDACT_EMAIL = True
REDACT_PHONE = True
REDACT_URL = True
DUP_SIMILARITY = 92
RANDOM_SEED = 42

### Project Folders

Create a tidy, repeatable project layout and set key thresholds.

**Folders**
- `data/processed/` - cleaned datasets for later notebooks
- `reports/` - audit artefacts (duplicates & quality metrics)

In [56]:
# Project Folders
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT/"data"
RAW_DIR = DATA_DIR/"raw"
PROCESSED_DIR = DATA_DIR/"processed"
MODELS_DIR = PROJECT_ROOT/"models"
FIGURES_DIR = PROJECT_ROOT/"figures"
ARTIFACTS_DIR = PROJECT_ROOT/"artifacts"
LOGS_DIR = PROJECT_ROOT/"logs"
REPORTS_DIR = PROJECT_ROOT/"reports"

for d in [DATA_DIR, RAW_DIR, PROCESSED_DIR, MODELS_DIR, FIGURES_DIR, ARTIFACTS_DIR, LOGS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

### Loading the Dataset

In [58]:
resume_df = pd.read_csv("hf://datasets/InferencePrince555/Resume-Dataset/updated_data_final_cleaned.csv")
resume_df.head()

,instruction,input,Resume_test
0,Generate a Resume for a Accountant Job,NaN,ACCOUNTANT Professional Summary Results orient...
1,Generate a Resume for a Accountant Job,NaN,STAFF ACCOUNTANT Summary Flexible Accountant w...
2,Generate a Resume for a Accountant Job,NaN,STAFF ACCOUNTANT Summary Highly analytical and...
3,Generate a Resume for a Accountant Job,NaN,SENIOR ACCOUNTANT Summary A highly competent m...
4,Generate a Resume for a Accountant Job,NaN,SENIOR ACCOUNTANT Summary 11 years experience ...


In [59]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32481 entries, 0 to 32480
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   instruction  32481 non-null  object 
 1   input        0 non-null      float64
 2   Resume_test  32480 non-null  object 
dtypes: float64(1), object(2)
memory usage: 761.4+ KB


In [60]:
resume_df.shape

(32481, 3)

## Select the Resume Text Column

Standardize schema. Your dataset typically has `instruction`, `input` and `Resume_test`.
We use **`Resume_test`** as the resume body.

**Output columns**
- `raw_text` - original resume text (string)
- `label` - placeholder for future labels (currently empty)

In [62]:
# Removing the unnecessary columns
resume_df.drop(['instruction', 'input'], axis=1, inplace=True)

In [63]:
resume_df.isnull().sum()

Resume_test    1
dtype: int64

In [64]:
resume_df = pd.DataFrame({
    "row_id": np.arange(len(resume_df), dtype=int),
    "raw_text": resume_df["Resume_test"].astype("string").fillna("")
})
resume_df["label"] = pd.NA
resume_df.head()

,row_id,raw_text,label
0,0,ACCOUNTANT Professional Summary Results orient...,<NA>
1,1,STAFF ACCOUNTANT Summary Flexible Accountant w...,<NA>
2,2,STAFF ACCOUNTANT Summary Highly analytical and...,<NA>
3,3,SENIOR ACCOUNTANT Summary A highly competent m...,<NA>
4,4,SENIOR ACCOUNTANT Summary 11 years experience ...,<NA>


## Cleaning Helpers

Define reusable functions for consistent text normalization and PII masking.

**Pipeline**
1. **Unicode normalize + transliterate** (`NFKC` -> ASCII via `unidecode`)
2. **Boilerplate strip**
3. **Whitespace tidy** (tabs -> spaces, bullet normalization, collapse runs)

**PII Redaction**
- `redact_basic` -> masks emails, phones, URLs with tokens
- `redact_ner` -> masks `PERSON` names via spaCy (if model is loaded)

> These steps reduce leakage and bias while preserving structure (bullets, paragraphs).

In [66]:
EMAIL_RE = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b')
PHONE_RE = re.compile(r'(?:(?:\+?\d{1,3}[\s\-\.]?)?(?:\(?\d{3}\)?[\s\-\.]?)?\d{3}[\s\-\.]?\d{4})')
URL_RE   = re.compile(r'(https?://\S+|www\.\S+)')

def normalize_unicode(s: str) -> str:
    return unidecode(unicodedata.normalize("NFKC", s))

def normalize_whitespace(s: str) -> str:
    s = s.replace("\t", " ")
    s = s.replace("•", "- ").replace(".", "- ").replace("◦", "- ")
    s = re.sub(r"[ \u00A0]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def strip_boilerplate(s:str) -> str:
    s = re.sub(r'\bpage\s*\d+\b', '', s, flags=re.I)
    s = re.sub(r'\bcontinued\b', '', s, flags=re.I)
    return s

def redact_basic(s: str, REDACT_EMAIL=True, REDACT_PHONE=True, REDACT_URL=True) -> str:
    if REDACT_EMAIL: s = EMAIL_RE.sub("[EMAIL]", s)
    if REDACT_PHONE: s = PHONE_RE.sub("[PHONE]", s)
    if REDACT_URL: s = URL_RE.sub("[URL]", s)
    return s

# spaCy PERSON redaction
def redact_ner(s: str) -> str:
    try:
        _ = _NLP
    except NameError:
        return s
    if not _NLP:
        return s
    out, last = [], 0
    doc = _NLP(s)
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            out.append(s[last:ent.start_char]); out.append("[PERSON]"); last = ent.end_char
    out.append(s[last:])
    return "".join(out)

def basic_clean(s: str) -> str:
    return normalize_whitespace(strip_boilerplate(normalize_unicode(s)))

def token_count(s: str) -> int:
    return len(s.split())

## Apply Cleaning & Redaction

Transform `raw_text` -> `clean_text` -> `clean_text_redacted`.

**Features added**
- `n_chars`, `n_tokens` for quick QA and filtering

In [68]:
ENABLE_REDACTION = True

resume_df["clean_text"] = resume_df["raw_text"].map(basic_clean)
resume_df["clean_text_redacted"] = (
    resume_df["clean_text"].map(lambda x: redact_basic(x, True, True, True)).map(redact_ner)
    if ENABLE_REDACTION else resume_df["clean_text"]
)

resume_df["n_chars"] = resume_df["clean_text_redacted"].str.len()
resume_df["n_tokens"] = resume_df["clean_text_redacted"].map(token_count)

resume_df[["raw_text", "clean_text_redacted", "n_tokens"]].head(5)

,raw_text,clean_text_redacted,n_tokens
0,ACCOUNTANT Professional Summary Results orient...,ACCOUNTANT Professional Summary Results orient...,746
1,STAFF ACCOUNTANT Summary Flexible Accountant w...,STAFF ACCOUNTANT Summary Flexible Accountant w...,1226
2,STAFF ACCOUNTANT Summary Highly analytical and...,STAFF ACCOUNTANT Summary Highly analytical and...,1051
3,SENIOR ACCOUNTANT Summary A highly competent m...,SENIOR ACCOUNTANT Summary A highly competent m...,659
4,SENIOR ACCOUNTANT Summary 11 years experience ...,SENIOR ACCOUNTANT Summary 11 years experience ...,673


In [69]:
resume_df["clean_text_redacted"]

0        ACCOUNTANT Professional Summary Results orient...
1        STAFF ACCOUNTANT Summary Flexible Accountant w...
2        STAFF ACCOUNTANT Summary Highly analytical and...
3        SENIOR ACCOUNTANT Summary A highly competent m...
4        SENIOR ACCOUNTANT Summary 11 years experience ...
                               ...                        
32476    Software Engineer span lSoftwarespan Engineer ...
32477    Sr Systems Manager Sr Business Manager Sr span...
32478    Full Stack NET Developer Full Stack NET span l...
32479    Director of Information Systems Director of In...
32480    UX Engineer UX Engineer UX Engineer Redmond WA...
Name: clean_text_redacted, Length: 32481, dtype: object

## Filter very short rows

Drop resumes below `MIN_CHARS` or `MIN_TOKENS`.
**Report:** The cell prints how many rows were removed and how many remain. 

In [71]:
before = len(resume_df)
resume_df = resume_df[(resume_df["n_chars"] >= MIN_CHARS) & (resume_df["n_tokens"] >= MIN_TOKENS)].copy()
after = len(resume_df)
print(f"[Filter] Removed {before - after} short rows. Remaining: {after}")

[Filter] Removed 21 short rows. Remaining: 32460


### Deduplicate (exact + optional near-duplicate)

Prevent training/test contamination and inflated metrics.

**Steps**
- **Exact:** normalize -> MD5 hash (`hash_norm`) -> keep first per hash
- **Near-dup (optional):** `rapidfuzz.token_set_ratio` on a sample; drop highly similar rows (`DUP_SIMILARITY`)

**Artefact:** `reports/duplicates.csv` lists duplicate groups for audit.

In [73]:
REPORTS_DIR = (Path.cwd() / "reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DUP_SIMILARITY, RANDOM_SEED = 92, 42

def norm_for_hash(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[\W_]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

resume_df["hash_norm"] = resume_df["clean_text_redacted"].map(norm_for_hash).map(lambda s: hashlib.md5(s.encode()).hexdigest())

dup_counts = resume_df["hash_norm"].value_counts()
dup_hashes = dup_counts[dup_counts > 1].index
dup_rows = resume_df[resume_df["hash_norm"].isin(dup_hashes)].sort_values(["hash_norm", "row_id"])
if len(dup_rows):
    dup_rows.to_csv(REPORTS_DIR / "duplicates.csv", index=False)
else:
    (REPORTS_DIR / "duplicates.csv").write_text("")

resume_df = resume_df.drop_duplicates(subset=["hash_norm"]).copy()
print(f"[Dedup] Exact-dup groups: {len(dup_hashes)}")

try:
    sample = resume_df[["row_id", "clean_text_redacted"]].sample(min(1000, len(df)), random_state=RANDOM_SEED)
    drop_ids = set()
    rows = list(sample.itertuples(index=False, name=None))
    for i in range(len(rows)):
        if rows[i][0] in drop_ids:
            continue
        for j in range(i+1, len(rows)):
            if rows[j][0] in drop_ids:
                continue
            if fuzz.token_set_ratio(rows[i][1], rows[j][1]) >= DUP_SIMILARITY:
                drop_ids.add(rows[j][0])
    if drop_ids:
        before_soft = len(resume_df)
        resume_df = resume_df[~resume_df["row_id"].isin(drop_ids)].copy()
        print(f"[Dedup] Soft near-dup removed ~{beofre_soft - len(resume_df)} rows.")
except Exception:
    pass

[Dedup] Exact-dup groups: 183


## Section Flags & Crude Skills Block

Light structure cues for EDA and later features.

**Adds**
- `has_skills`, `has_education`, `has_experience` - simple substring checks
- `skills_block` - grabs bullet-like lines under a "Skills" header up to next section

> These features help with quick distibution checks and can seed simple baselines.

In [75]:
def has_section(s: str, key: str) -> bool:
    return key in s.lower()

for hdr in ["skills", "education", "experience"]:
    resume_df[f"has_{hdr}"] = resume_df["clean_text_redacted"].map(lambda s, h=hdr: has_section(s, h))

def extract_skills_block(text: str) -> str:
    t = text.lower()
    start = None
    for hdr in ["skills", "technical skills", "key skills"]:
        pos = t.find(hdr)
        if pos != -1:
            start = pos + len(hdr); break

    if start is None: return ""
    block = text[start:]
    for stop in ["experience", "work history", "employment history", "projects", "education", "certifications", "publications"]:
        p = block.lower().find(stop)
        if p != 1:
            block = block[:p]; break
    lines = [ln.strip("-• \t") for ln in block.splitlines() if ln.strip()]
    lines = [ln for ln in lines if len(ln) <= 80]
    return "; ".join(lines[:100])

resume_df["skills_block"] = resume_df["clean_text_redacted"].map(extract_skills_block)

## Saving cleaned outputs and Data Quality report

Persist a clean, analysis-ready dataset

**Files:** `data/processed/resumes_clean.csv` (+ `.parquet`
CSV is universal; Parquet is faster and preserves types for modeling.

**Data Quality Report Metrics**
- Row count, avg characters/tokens
- Label presence/cardinality
- Token length buckets (for histogram)
- Fractions with skills/education/experience sections
  
**File:** `reports/data_quality.json`

In [77]:
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path.cwd() / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

out = resume_df.drop(columns=["hash_norm"], errors="ignore").copy()

clean_csv = PROCESSED_DIR / "resumes_clean.csv"
clean_parquet = PROCESSED_DIR / "resumes_clean.parquet"
out.to_csv(clean_csv, index=False)

try:
    out.to_parquet(clean_parquet, index=False)
    print(f"[Save] {clean_csv} and {clean_parquet}")
except Exception as e:
    print(f"[Save] CSV only: {clean_csv} (Parquet optional). Error: {e}")

buckets = pd.cut(out["n_tokens"], bins=[0,50,100,200,400,800,1600,1e9], right=False).value_counts().sort_index()
quality = {
    "n_rows": int(len(out)),
    "avg_chars": float(out["n_chars"].mean()),
    "avg_tokens": float(out["n_tokens"].mean()),
    "has_label": int(out["label"].notna().sum()),
    "label_cardinality": int(out["label"].nunique(dropna=True)),
    "label_top10": out["label"].value_counts(dropna=True).head(10).to_dict(),
    "token_buckets": {str(k): int(v) for k, v in buckets.items()},
    "frac_has_skills": float(out["has_skills"].mean()),
    "frac_has_education": float(out["has_education"].mean()),
    "frac_has_experience": float(out["has_experience"].mean()),
}
with open(REPORTS_DIR / "data_quality.json", "w") as f:
    json.dump(quality, f, indent=2)

print("[Done] Cleaned data ->", clean_csv)
print("[Done] Quality report ->", REPORTS_DIR / "data_quality.json")

[Save] C:\Users\Keya Barua\Documents\MSC Data Science - Middlesex University\CST 4090\data\processed\resumes_clean.csv and C:\Users\Keya Barua\Documents\MSC Data Science - Middlesex University\CST 4090\data\processed\resumes_clean.parquet
[Done] Cleaned data -> C:\Users\Keya Barua\Documents\MSC Data Science - Middlesex University\CST 4090\data\processed\resumes_clean.csv
[Done] Quality report -> C:\Users\Keya Barua\Documents\MSC Data Science - Middlesex University\CST 4090\reports\data_quality.json
